In [1]:
print(123)

123



## Question 1. Instrument the agent with Logfire

Sign up for a free [Logfire](https://logfire.dev) account, create a
project, and generate a write token. Put it in `.env` as
`LOGFIRE_TOKEN`.

Instrument the agent:

```python
logfire.configure()
logfire.instrument_pydantic_ai()
```

Run the agent a few times with different questions and open your
project on Logfire to see the traces.

For the following query

> How do I run Ollama locally?

how many spans does a single agent run produce?

Each span is either the agent run itself, an LLM call, or a tool call.
The number can vary between runs because the model decides how many
times to search.

* 1
* 5
* 15
* 30

ANS:5

In [13]:
## following code run from python
!uv run python main_instrumented.py

Logfire project URL: https://logfire-us.pydantic.dev/jobwils2025/starter-project
06:52:19.313 faq_agent run
06:52:19.314   chat gpt-5.4-mini
06:52:21.510   running tool: search
06:52:21.517   chat gpt-5.4-mini
You can run Ollama locally like this:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model locally:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model and opens a chat-like interface.

3. Test that the local server is running:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]} 
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content


## Question 2. Load traces into DuckDB with dlt

Generate a read token for your Logfire project and set it as
`LOGFIRE_READ_TOKEN` in `.env`.

Initialize a dlt-hub project like in the workshop. Then ask your coding
agent to pull the data from Pydantic Logfire and save it into DuckDB.

The dltHub AI workbench has a ready-made context for Logfire. Point your
agent to it: https://dlthub.com/context/source/logfire

If you don't currently use a coding agent, you can use something like OpenCode:
you should be able to complete one session with the free account.

Alternatively, you can do it in the old way (using ChatGPT or your favorite search engine).

If you don't currently use a coding agent, you can use something like OpenCode:
you should be able to complete one session with the free account. 

Alternatively, you can do it in the old way (using ChatGPT or your favorite search engine).

The logfire traces contain deeply nested JSON (span attributes with
LLM messages, tool calls, token usage, etc.). dlt automatically
normalizes this into a set of tables - one for the main records, plus
child tables for each nested level.

How many tables did dlt create? Check with:

```sql
SELECT COUNT(*) FROM information_schema.tables 
WHERE table_schema = 'agent_traces';
```

* 1
* 3
* 24
* 100

ANS: 24 (closest to 27)

In [3]:
import dlt
print(dlt.__version__)

1.29.1


In [5]:
import os

import dlt
from dotenv import load_dotenv
from logfire.query_client import AsyncLogfireQueryClient

load_dotenv()  # Loads LOGFIRE_READ_TOKEN from .env

@dlt.resource(
    name="records",
    write_disposition="replace",
)
async def logfire_records():
    async with AsyncLogfireQueryClient(
        os.environ["LOGFIRE_READ_TOKEN"]
    ) as client:
        rows = await client.query_json(
            sql="SELECT * FROM records"
        )

        # dlt will recursively normalize the nested JSON
        yield rows

pipeline = dlt.pipeline(
    pipeline_name="logfire",
    destination="duckdb",
    dataset_name="agent_traces",
)

load_info = pipeline.run(logfire_records())
load_info

/tmp/ipykernel_5443/2648560023.py:17: DeprecationWarning: Querying without a min_timestamp is deprecated
  rows = await client.query_json(
2026-07-31 06:28:58,941|[WARNING]|5443|124871735236416|dlt|validate.py|verify_normalized_table:113|In schema `logfire`: The following columns in table 'records__columns__values' did not receive any data during this load and therefore could not have their types inferred:
  - model_request_parameters__output_object
  - model_request_parameters__prompted_output_template
  - model_request_parameters__thinking

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'model_request_parameters__output_object': {'data_type': 'text'}})

2026-07-31 06:28:58,942|[WARNING]|5443|124871735236416|dlt|validate.py|verify_normalized_table:113|In schema `logfire`: The following columns in table 'recor

LoadInfo(pipeline=<dlt.pipeline(pipeline_name='logfire', destination='duckdb', dataset_name='agent_traces', default_schema_name='logfire', schema_names=['logfire'], first_run=False, dev_mode=False, is_active=True, pipelines_dir='/home/jobin/.dlt/pipelines', working_dir='/home/jobin/.dlt/pipelines/logfire')>, metrics={'1785459537.8545363': [{'started_at': DateTime(2026, 7, 31, 0, 58, 59, 7279, tzinfo=Timezone('UTC')), 'finished_at': DateTime(2026, 7, 31, 0, 58, 59, 515349, tzinfo=Timezone('UTC')), 'job_metrics': {'records__columns__values__gen_ai_tool_call_result.c3b2197b10.insert_values.gz': LoadJobMetrics(job_id='records__columns__values__gen_ai_tool_call_result.c3b2197b10.insert_values.gz', file_path='/home/jobin/.dlt/pipelines/logfire/load/normalized/1785459537.8545363/started_jobs/records__columns__values__gen_ai_tool_call_result.c3b2197b10.0.insert_values.gz', table_name='records__columns__values__gen_ai_tool_call_result', started_at=DateTime(2026, 7, 31, 0, 58, 59, 215499, tzinfo

In [6]:
import duckdb

conn = duckdb.connect("logfire.duckdb")

conn.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='agent_traces'
ORDER BY table_name
""").show()

┌──────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                              table_name                                              │
│                                               varchar                                                │
├──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ _dlt_loads                                                                                           │
│ _dlt_pipeline_state                                                                                  │
│ _dlt_version                                                                                         │
│ records                                                                                              │
│ records__columns                                                                                     │
│ records__columns__datatype__timestamp                

In [7]:

sql_query = """SELECT COUNT(*) FROM information_schema.tables 
WHERE table_schema = 'agent_traces'"""
conn.sql(sql_query).show()


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           27 │
└──────────────┘



## Question 3. Query traces with an agent

Using a coding agent (you can also write the code by hand) find the
input token usage for the agent run from Q1.

The token counts are stored in the span attributes as
`gen_ai.usage.input_tokens`. Sum them across all LLM calls within the
trace. The number depends on how many searches the agent made, so
report the range it falls into:

* 100 - 500
* 1500 - 5000
* 10000 - 20000
* 50000 - 100000

ANS: 1500 - 5000

In [8]:
table_details_sql =  """SELECT
    table_name,
    column_name
FROM information_schema.columns
WHERE table_schema = 'agent_traces'
  AND column_name ILIKE '%input%'
ORDER BY table_name, column_name"""
conn.sql(table_details_sql).show()


┌──────────────────────────┬─────────────────────────────────────────────────┐
│        table_name        │                   column_name                   │
│         varchar          │                     varchar                     │
├──────────────────────────┼─────────────────────────────────────────────────┤
│ records__columns__values │ gen_ai_aggregated_usage_cache_read_input_tokens │
│ records__columns__values │ gen_ai_aggregated_usage_input_tokens            │
│ records__columns__values │ gen_ai_usage_cache_read_input_tokens            │
│ records__columns__values │ gen_ai_usage_input_tokens                       │
│ records__columns__values │ properties__gen_ai_input_messages__type         │
└──────────────────────────┴─────────────────────────────────────────────────┘



In [12]:
trace_grqoup_sql = """describe 
agent_traces.records__columns__values"""
conn.sql(trace_grqoup_sql).show()

┌──────────────────────────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│                 column_name                  │       column_type        │  null   │   key   │ default │  extra  │
│                   varchar                    │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ value                                        │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ _dlt_parent_id                               │ VARCHAR                  │ NO      │ NULL    │ NULL    │ NULL    │
│ _dlt_list_idx                                │ BIGINT                   │ NO      │ NULL    │ NULL    │ NULL    │
│ _dlt_id                                      │ VARCHAR                  │ NO      │ NULL    │ NULL    │ NULL    │
│ value__v_text                                │ VARCHAR                

In [14]:
trace_grqoup_sql = """SELECT
    _dlt_id, gen_ai_conversation_id,gen_ai_usage_input_tokens
FROM agent_traces.records__columns__values
WHERE gen_ai_usage_input_tokens IS NOT NULL"""
conn.sql(trace_grqoup_sql).show()

┌────────────────┬──────────────────────────────────────┬───────────────────────────┐
│    _dlt_id     │        gen_ai_conversation_id        │ gen_ai_usage_input_tokens │
│    varchar     │               varchar                │           int64           │
├────────────────┼──────────────────────────────────────┼───────────────────────────┤
│ F65KWYYKHUBM8Q │ 019fb5a5-2169-74a5-a277-027c9aa57c42 │                      2520 │
│ Azs/hErV7OOr7w │ 019fb5a5-2169-74a5-a277-027c9aa57c42 │                      1419 │
│ kIObIYxR4vaxJw │ 019fb5a5-2169-74a5-a277-027c9aa57c42 │                       204 │
│ sKrgrSjWOmk6vQ │ 019fb097-2495-71ea-9d12-7365c38c268e │                      1282 │
│ aBbIGMXAzLAh6g │ 019fb097-2495-71ea-9d12-7365c38c268e │                       204 │
│ IuHI5xVpPaLyOw │ 019f9d0d-7f9e-7395-9a22-983f6a91de79 │                       825 │
│ 7qPlREb4Dw/7zQ │ 019f9d0d-7f9e-7395-9a22-983f6a91de79 │                       207 │
│ 5w+/WhU6tnbcAw │ 019fa116-41bc-72d4-9931-f8e262cb996